# Заняття 5 · Демо 2 · CLIP і SigLIP 2: спільний простір, zero-shot і де він ламається

**LIVE DEMO 2 (слайд 40)** · слайди 22, 25, 29, 34–39, 46, 48–51

Розбираємо по шматках, що всередині мультимодальної моделі: дві вежі, косинус, температура, softmax проти сигмоїди,
промпти й типові відмови. Порівнюємо дві моделі з **однаковим бекбоном ViT-B/16 і входом 224 px** — відрізняються
вони даними, токенайзером і функцією втрат:

| | OpenAI CLIP (2021) | SigLIP 2 (Google, 2025) |
|---|---|---|
| Дані | 400 млн вебових пар | WebLI, мультимовні, з фільтрацією |
| Втрата | softmax InfoNCE по батчу | сигмоїда на кожну пару (+ captioning, self-distillation) |
| Вихід | лише **відносний** ранг між промптами | ще й **абсолютна** оцінка пари «кадр–текст» |

| Розділ | Що показуємо | Слайди |
|---|---|---|
| 1. Дві вежі | Що в чекпойнті: розмірність, токенайзери, `logit_scale`, `logit_bias`; пастка QuickGELU | 34, 35 |
| 2. Спільний простір | Матриця cos «кадр × текст» | 34 |
| 3. Контрастивна втрата | InfoNCE проти сигмоїдної, температура (повзунок) | 35 |
| 4. Zero-shot | Промпти з ракурсом, довірчий інтервал, клас «фон» | 36, 38 |
| 5. Пісочниця | Свої промпти, напис у кадрі, поворот, інверсія ІЧ | 35–39 |
| 6. Softmax ≠ впевненість | Що бачить оператор, якщо показати йому число з моделі | 37, 48, 49 |
| 7. Батарея відмов | Заперечення, рахунок, напис, поворот, полярність ІЧ — як регресійний тест | 37–39 |
| 8. Текст чи мітки | Zero-shot проти DINOv2 з 1–20 мітками на клас | 29, 34 |

**Підготовка (один раз, з мережею):**
```bash
# з кореня репозиторію
python tools/fetch_clip_models.py --models clip siglip2   # ~2.1 ГБ -> artifacts/models/clip/
python tools/fetch_visdrone.py --splits train val test-dev
python tools/fetch_dino_models.py --models dinov2          # лише для розділу 8
# для SIZE = "large" додайте --size large до обох команд моделей
```
Усе рахується за 1–2 хвилини на GPU; на CPU — до ~15 хв (зменште `PER_CLASS` і `FRAMES_PER_SEQ`).

In [ ]:
import os
from pathlib import Path

# ---------------- Конфігурація ----------------
DEVICE = None          # None — cuda / mps / cpu автоматично
SIZE = "base"          # "base": CLIP ViT-B/16 vs SigLIP 2 ViT-B/16 | "large": CLIP ViT-L/14 vs SigLIP 2 ViT-L/16-256
MODELS = ["clip", "siglip2"]
PER_CLASS = 80         # кропів на клас VisDrone (ціль ≥ 32 px)
N_BACKGROUND = 80      # кропів «фону» без жодного об'єкта
FRAMES_PER_SEQ = 6     # цілих кадрів на послідовність для батареї відмов (розділ 7)
USE_DINO = True        # розділ 8: потрібні ваги DINOv2
SEED = 0
CACHE_DIR = Path("outputs/demo2")

import panels as P
hf = P.setup_hf(clip=MODELS, dino=["dinov2"] if USE_DINO else [], size=SIZE)   # офлайн, якщо всі ваги вже в маніфестах

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from PIL import Image
pd.set_option("display.precision", 3)
rng = np.random.default_rng(SEED)

P.ensure_visdrone(["train", "val", "test-dev"])
seqs = P.list_sequences()
LABELS = list(P.CLASSES)       # pedestrian, bicycle, car, van, truck, tricycle, bus, motorcycle
obj = P.extract_crops(seqs, min_px=32, max_occ=0, cap_per_class_seq=4, seed=SEED)
obj = obj.sample(frac=1, random_state=SEED).groupby("cls").head(PER_CLASS)   # до PER_CLASS кропів на клас
crops = P.materialize_crops(pd.concat([obj, P.background_crops(seqs, N_BACKGROUND, seed=SEED)], ignore_index=True),
                            CACHE_DIR / "crops")
crop_imgs = P.load_images(crops.crop_path)
y_all = crops.cls.to_numpy()
bg = y_all == "background"
print(f"HF: {hf} | кропів: {len(crops)} {crops.cls.value_counts().to_dict()}")
print(f"коротка сторона цілі: медіана {crops[~bg].short.median():.0f} px")

## 1. Дві вежі: що лежить у чекпойнті

CLIP-подібна модель — це **дві незалежні вежі**: image encoder (ViT) і text encoder. Після L2-нормалізації
скалярний добуток двох векторів дорівнює косинусу (слайд 34 «Спільний простір»):

$$v = \frac{f_I(I)}{\lVert f_I(I)\rVert},\quad u = \frac{f_T(T)}{\lVert f_T(T)\rVert},\quad \mathrm{sim}(I,T) = v^\top u = \cos\theta$$

Крім ваг веж, у чекпойнті є ще одне-два числа, які знадобляться в інференсі (слайд 35 «Контрастивна функція втрат»):
`logit_scale` = 1/τ — температура, що робить softmax по косинусах різким, і `logit_bias` — зсув сигмоїди, є **лише в SigLIP**.

In [ ]:
import open_clip

enc = {m: P.ClipEncoder(m, SIZE, device=DEVICE) for m in MODELS}


def describe(e):
    m, tcfg = e.model, open_clip.get_model_config(e.arch)["text_cfg"]
    tok = m.token_embedding if hasattr(m, "token_embedding") else m.text.token_embedding
    return {"архітектура (open_clip)": e.arch, "параметрів, млн": round(e.n_params / 1e6),
            "  з них таблиця ембеддингів токенів, млн": round(tok.weight.numel() / 1e6),
            "розмір ембеддингу": e.dim, "вхід image tower": f"{e.input_size[0]}×{e.input_size[1]}",
            "нормалізація пікселів (mean)": tuple(round(x, 3) for x in e.mean),
            "контекст тексту, токенів": tcfg.get("context_length", 77), "словник токенайзера": tcfg.get("vocab_size", 49408),
            "logit_scale = 1/τ": round(e.logit_scale, 1), "logit_bias": "—" if e.logit_bias is None else round(e.logit_bias, 2)}


def tokens(e, text):
    ids, tok = e.tokenizer([text])[0].tolist(), e.tokenizer
    if hasattr(tok, "tokenizer"):                                   # SigLIP 2: HF-токенайзер Gemma
        return tok.tokenizer.convert_ids_to_tokens([i for i in ids if i != tok.tokenizer.pad_token_id])
    return [tok.decoder[i].replace("</w>", "_") for i in ids if i != 0]   # CLIP: BPE, 0 = паддінг


display(pd.DataFrame({k: describe(e) for k, e in enc.items()}))
for k, e in enc.items():
    t = tokens(e, "An aerial photo of a white TRUCK near a building!")
    print(f"{k:8s} {len(t):2d} токенів (з {e.tokenizer.context_length}): {t}")

Що помітити:
- **Різна розмірність простору** (512 проти 768): вектори двох моделей не порівнюються, а індекс, побудований однією
  моделлю, з текстовим енкодером іншої не працює (слайд 51 «Split-computing»: обидві вежі — з однієї моделі).
- **SigLIP 2 «важчий» здебільшого через словник** Gemma на 256 тис. токенів; image tower в обох — ViT-B/16 (~86 млн),
  тож на борту, де потрібна лише image tower, різниця майже зникає.
- **Різна нормалізація пікселів**: препроцесинг завжди беремо з тієї самої моделі — з чужим вона не впаде, просто працюватиме гірше.
- **Токенайзери мовчки змінюють промпт.** CLIP переводить текст у нижній регістр, але лишає `!` окремим токеном; SigLIP 2
  канонікалізує (без регістру й пунктуації). Контекст — 77 і 64 токени: довгий опис обрізається, важливе пишемо на початку.

### Пастка: QuickGELU

Ваги OpenAI CLIP навчено з активацією `QuickGELU`, а open_clip 3.x для `ViT-B-16` за замовчуванням ставить звичайний `GELU`
і лише попереджає. Модель працює, ембеддинги осмислені — просто трохи гірші. У `P.CLIP_ZOO` для CLIP стоїть `quick_gelu=True`;
порівняємо з «неправильною» версією на zero-shot по кропах (метрика — розділ 4).

In [ ]:
import warnings
spec = P.CLIP_ZOO["clip"][SIZE]
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    bad = open_clip.create_model(spec["arch"], pretrained=spec["pretrained"], cache_dir=str(P.CLIP_DIR / "open_clip"))
good = enc["clip"]
bad = bad.to(good.device).eval()
captions = [f"an aerial photo of a {l}" for l in LABELS]
with torch.no_grad():
    x = torch.from_numpy(good.prepare_images([crop_imgs[i] for i in np.flatnonzero(~bg)])).to(good.device)
    x = (x.permute(0, 3, 1, 2).float() / 255 - good._mean) / good._std
    Ib = torch.nn.functional.normalize(bad.encode_image(x).float(), dim=-1).cpu().numpy()
    Tb = torch.nn.functional.normalize(bad.encode_text(good.tokenizer(captions).to(good.device)).float(), dim=-1).cpu().numpy()
Ig, Tg = good.encode_images([crop_imgs[i] for i in np.flatnonzero(~bg)]), good.encode_texts(captions)
for name, I_, T_ in [("QuickGELU (правильно)", Ig, Tg), ("GELU (тихо неправильно)", Ib, Tb)]:
    print(f"{name:24s} macro-recall = {P.macro_recall(y_all[~bg], np.array(LABELS)[(I_ @ T_.T).argmax(1)]):.3f}")
del bad, x

**Підсумок розділу.** Простір, розмірність, препроцесинг і токенайзер прив'язані до конкретного чекпойнта. Помилка в будь-чому
з цього не валить модель, а тихо знижує якість — перевіряйте на своїх даних після кожної зміни.

## 2. Спільний простір: матриця «кадр × текст»

По одному кропу чотирьох класів VisDrone і чотири описи. На діагоналі мають бути найбільші значення в рядку.

In [ ]:
demo_labels = ["car", "bus", "pedestrian", "bicycle"]
demo_imgs = [crop_imgs[int(rng.choice(np.flatnonzero(y_all == l)))] for l in demo_labels]
demo_texts = [f"an aerial photo of a {l}" for l in demo_labels]
S = {k: e.encode_images(demo_imgs) @ e.encode_texts(demo_texts).T for k, e in enc.items()}
P.show_matrix(S, demo_imgs, demo_texts, titles={k: f"{k}: cos ∈ [{m.min():.2f}, {m.max():.2f}]" for k, m in S.items()})

**Підсумок розділу.** Абсолютні косинуси малі й лежать у вузькому діапазоні (~0.2–0.35 у CLIP, ~0.0–0.2 у SigLIP 2):
кадри й тексти займають різні «конуси» сфери (modality gap, Демо 3). Ранжування працює, бо важить різниця, а не значення.
Шкали моделей різні, тож поріг однієї моделі не переноситься на іншу (слайд 49).

## 3. Контрастивна втрата і температура

Батч із N = 8 пар «кроп — підпис», обидві втрати вручну з тими `logit_scale` / `logit_bias`, з якими модель навчалась (слайд 35).

**CLIP, симетричний InfoNCE:** кожен рядок і стовпець — softmax-класифікація «яка з N пар моя»:
$\mathcal{L}_{\text{CLIP}} = \tfrac12\big[\mathrm{CE}(sVU^\top, \mathrm{diag}) + \mathrm{CE}(sUV^\top, \mathrm{diag})\big]$

**SigLIP, сигмоїда на пару:** кожна клітинка — окрема бінарна задача, нормалізації по батчу немає:
$\mathcal{L}_{\text{SigLIP}} = -\frac1N\sum_{i,j}\log\sigma\big(z_{ij}(s\,v_i^\top u_j + b)\big)$, $z_{ij}=+1$ на діагоналі, $-1$ поза нею.

In [ ]:
import torch.nn.functional as F

batch_imgs = [crop_imgs[int(rng.choice(np.flatnonzero(y_all == l)))] for l in LABELS]
mats, titles = {}, {}
for k, e in enc.items():
    s, b = e.logit_scale, e.logit_bias
    cos = torch.from_numpy(e.encode_images(batch_imgs) @ e.encode_texts(captions).T)
    y = torch.arange(len(LABELS))
    if b is None:
        mats[k] = (s * cos).softmax(dim=1).numpy()                     # рядок — розподіл по N підписах, сума = 1
        loss = 0.5 * (F.cross_entropy(s * cos, y) + F.cross_entropy(s * cos.T, y))
        titles[k] = f"{k}: softmax по рядку (s={s:.0f}) · InfoNCE = {loss:.3f}"
    else:
        mats[k] = torch.sigmoid(s * cos + b).numpy()                   # кожна клітинка незалежно
        loss = -F.logsigmoid((2 * torch.eye(len(LABELS)) - 1) * (s * cos + b)).sum() / len(LABELS)
        titles[k] = f"{k}: sigmoid(s·cos + b) (b={b:.1f}) · loss = {loss:.3f}"
    print(f"{k:8s} суми рядків: {np.round(mats[k].sum(1), 2)}")
P.show_matrix(mats, [f"кроп: {l}" for l in LABELS], LABELS, fmt=".2f", cmap="magma", vmin=0, vmax=1, titles=titles)

Рядок CLIP завжди дає в сумі 1, навіть якщо жоден підпис не підходить. Клітинки SigLIP незалежні: сума рядка може бути ≈ 0
або > 1. Тому SigLIP не потребує батчу на 32 768 і дає число, осмислене для однієї пари.

**Температура.** Сирі косинуси відрізняються в другому знаку; рухайте `1/τ` і дивіться, як softmax із майже рівномірного
стає різким. Якщо в інференсі забути `logit_scale`, ранг не зміниться, але «відсотки впевненості» стануть сміттям.

In [ ]:
from ipywidgets import interact, FloatLogSlider

e = enc["clip"]
cos_bus = (e.encode_images([batch_imgs[LABELS.index("bus")]]) @ e.encode_texts(captions).T)[0]


@interact(s=FloatLogSlider(value=e.logit_scale, base=10, min=0, max=2.5, step=0.05, description="1/τ", continuous_update=False))
def temperature(s):
    p = torch.softmax(torch.from_numpy(s * cos_bus), 0).numpy()
    plt.figure(figsize=(8, 2.4))
    plt.bar(LABELS, p, color=["tab:orange" if l == "bus" else "tab:gray" for l in LABELS])
    plt.ylim(0, 1)
    plt.title(f"CLIP, кроп «bus»: softmax(s·cos), s = {s:.1f} (навчене {e.logit_scale:.0f}, старт навчання {1 / 0.07:.0f})", fontsize=9)
    plt.xticks(fontsize=8)
    plt.show()

**Підсумок розділу.** InfoNCE вчить **ранжувати** пари всередині батчу, сигмоїдна втрата — **оцінювати** кожну пару окремо.
Температура — частина моделі: без `logit_scale` (і `logit_bias` у SigLIP) ймовірності не мають сенсу.

## 4. Zero-shot на кропах VisDrone

Класичний рецепт (слайд 36 «CLIP: Zero-Shot Classification»): клас → текст → вектор, перемагає найближчий. Варіанти промптів (слайд 38 «Доменний розрив ракурсу»):
`plain` — `"a truck"`; `photo` — `"a photo of a truck."`; `aerial×6` — шість формулювань із ракурсом, вектори усереднюються.
Метрика — **macro-recall** (слайд 46) і бутстреп-інтервал: на ~600 кропах кілька пунктів різниці можуть бути шумом.

In [ ]:
PROMPT_SETS = {"plain": P.PLAIN_TEMPLATES, "photo": P.PHOTO_TEMPLATES, "aerial×6": P.AERIAL_TEMPLATES}
IMG = {k: e.encode_images(crop_imgs) for k, e in enc.items()}
y_obj = y_all[~bg]


def class_text(e, labels, templates):
    return np.stack([e.text_query(f"a {l}", templates) for l in labels])


PRED, rows = {}, []
for k, e in enc.items():
    for pname, tpls in PROMPT_SETS.items():
        PRED[k, pname] = np.array(LABELS)[(IMG[k][~bg] @ class_text(e, LABELS, tpls).T).argmax(1)]
        rec = {l: (PRED[k, pname][y_obj == l] == l).mean() for l in LABELS}
        rows.append(dict(model=k, prompts=pname, macro_recall=np.mean(list(rec.values())), **rec))
display(pd.DataFrame(rows).set_index(["model", "prompts"]).style.format("{:.2f}").background_gradient(cmap="RdYlGn", vmin=0, vmax=1))

for pname in PROMPT_SETS:
    lo, med, hi = P.bootstrap_diff(y_obj, PRED["siglip2", pname], PRED["clip", pname], seed=SEED)
    print(f"{pname:9s} SigLIP 2 − CLIP = {med:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}] → {'значуща' if lo > 0 or hi < 0 else 'у межах шуму'}")
for k in MODELS:
    lo, med, hi = P.bootstrap_diff(y_obj, PRED[k, "aerial×6"], PRED[k, "plain"], seed=SEED)
    print(f"{k:8s}  aerial×6 − plain = {med:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]")
P.plot_confusions({f"{k} · aerial×6": P.confusion(y_obj, PRED[k, "aerial×6"], LABELS) for k in MODELS}, LABELS)

Типові плутанини — `van ↔ car`, `truck ↔ van`, `bicycle ↔ motorcycle`: на 40 px їх важко розрізнити й людині, і частина
«помилок» — спірна розмітка (пікап, підписаний `truck`). Це питання пікселів і таксономії, а не архітектури (слайди 22, 25).

### Клас «фон»

Закритий список із 8 класів **змушує** назвати кожен кроп трави «car» або «truck». Додаємо фонові промпти — максимум по кількох
описах (фон неоднорідний, середнє «ні про що» гірше) — і дивимось на ціну: скільки фону відсіюється і скільки цілей губиться.

In [ ]:
BG_PHRASES = ["an empty road", "grass", "trees", "the roof of a building", "a parking lot with no cars", "water", "a sidewalk"]
rows = []
for k, e in enc.items():
    s_obj = IMG[k] @ class_text(e, LABELS, P.AERIAL_TEMPLATES).T
    s_bg = (IMG[k] @ np.stack([e.text_query(p, P.AERIAL_TEMPLATES) for p in BG_PHRASES]).T).max(1, keepdims=True)
    for with_bg in (False, True):
        pred = np.array(LABELS + ["background"] * with_bg)[(np.hstack([s_obj, s_bg]) if with_bg else s_obj).argmax(1)]
        rows.append({"model": k, "фонові промпти": "так" if with_bg else "ні",
                     "macro-recall об'єктів": P.macro_recall(y_obj, pred[~bg]),
                     "фон → «об'єкт»": (pred[bg] != "background").mean(), "об'єкт → «фон»": (pred[~bg] == "background").mean()})
pd.DataFrame(rows).set_index(["model", "фонові промпти"])

**Підсумок розділу.**
* Ракурс у промпті — дешевий приріст, і різним моделям він допомагає по-різному. «Яка модель краща» залежить від того,
  як із нею говорити: порівнюйте в однакових умовах і з довірчим інтервалом.
* Без класу «фон» 100 % порожніх кропів стають «об'єктом». Фонові промпти відсіюють більшість фону, але на цілих кадрах,
  де ціль займає 1 % площі, вони «з'їдатимуть» і справжні цілі — це той самий компроміс, що й поріг `T_low` (слайд 49).

## 5. Пісочниця: свої промпти, напис, поворот, полярність ІЧ

Оберіть кадр, змініть промпти (по одному на рядок) і модифікації. Ліворуч — сирий cos, праворуч — те, що модель видала б як
«ймовірність»: у CLIP — softmax **між переліченими промптами**, у SigLIP 2 — сигмоїда **для кожного окремо**.

**Що спробувати:**
* кадр із танками + напис `AMBULANCE` або `SCHOOL BUS` — найдешевша атака (слайд 37 «Де CLIP помиляється»);
* додати промпт `an aerial photo of a road without cars` до кадру з машинами — заперечення (слайд 37);
* прибрати з переліку правильний клас: softmax CLIP однаково «впевнено» вибере щось інше (слайд 37, «некалібрована впевненість»);
* поворот 90° / 180° — чутливість до азимута (слайд 38); LWIR і інверсія — тепловізор поза доменом (слайд 39 «Тепловізор»).

In [ ]:
rgb, lwir = (Image.open(P.IMAGES_DIR / f).convert("RGB") for f in ("lesson_1_demo_1_rgb.jpg", "lesson_1_demo_1_lwir.jpg"))
TANK_BOX = (1987, 773, 2346, 1049)        # px кадру 2752×1536: три танки праворуч (ілюстративна пара RGB / LWIR заняття 1)
car_row = crops[crops.cls == "car"].sort_values("short").iloc[-1]   # найбільша машина у вибірці — для тесту рахунку
car = Image.open(car_row.path).convert("RGB").crop((car_row.x, car_row.y, car_row.x + car_row.w, car_row.y + car_row.h))
PLAY = {"танки, RGB": rgb.crop(TANK_BOX), "танки, LWIR white-hot": lwir.crop(TANK_BOX), "3 машини (синтетика)": P.paste_copies(car, 3)}
for l in ["car", "bus", "pedestrian", "truck"]:
    PLAY[f"VisDrone: {l}"] = crop_imgs[int(rng.choice(np.flatnonzero(y_all == l)))]
PLAY["VisDrone: фон"] = crop_imgs[int(rng.choice(np.flatnonzero(bg)))]


def zs_scores(m, image, prompts):
    e = enc[m]
    cos = e.encode_images([image])[0] @ e.encode_texts(prompts).T
    if e.logit_bias is None:
        return dict(cos=cos, prob=torch.softmax(torch.from_numpy(e.logit_scale * cos), 0).numpy(), prob_name="softmax між промптами")
    return dict(cos=cos, prob=torch.sigmoid(torch.from_numpy(e.logit_scale * cos + e.logit_bias)).numpy(), prob_name="sigmoid, кожен окремо")


P.zero_shot_playground(PLAY, zs_scores, MODELS, image="танки, RGB", prompts=[
    "an aerial photo of tanks", "an aerial photo of an ambulance", "an aerial photo of a school bus",
    "an aerial photo of cars", "a thermal image of tanks", "an aerial photo of an empty field"])

**Підсумок розділу.** Модель чесно відповідає лише на питання «який із цих описів ближчий». Напис у кадрі, відсутній у переліку
правильний клас, поворот чи інша полярність ІЧ змінюють відповідь, а «впевненість» softmax при цьому може лишатися високою.

## 6. Softmax ≠ впевненість

«Модель упевнена на 94 %» — найнебезпечніше число на екрані оператора (слайд 48 «Human-in-the-loop»). Порівнюємо на всіх кропах
максимум softmax CLIP по 8 класах і максимум сигмоїди SigLIP 2 — окремо для кропів з об'єктом і порожнього фону.

In [ ]:
panels = []
for k, e in enc.items():
    cos = torch.from_numpy(IMG[k] @ class_text(e, LABELS, P.AERIAL_TEMPLATES).T)
    if e.logit_bias is None:
        conf, name = (e.logit_scale * cos).softmax(1).max(1).values.numpy(), "CLIP: max softmax по 8 класах"
    else:
        conf, name = torch.sigmoid(e.logit_scale * cos + e.logit_bias).max(1).values.numpy(), "SigLIP 2: max sigmoid(s·cos + b)"
    thr = np.percentile(conf[~bg], 25)
    print(f"{k:8s} фону з «впевненістю» вищою за найслабшу чверть об'єктів ({thr:.2f}): {(conf[bg] > thr).mean():.0%}")
    panels.append(dict(pos=conf[~bg], neg=conf[bg], title=name, pos_label="кропи з об'єктом", neg_label="фон (нічого немає)"))
P.plot_hist_pair(panels, "«впевненість» моделі", "Що бачить оператор, якщо показати йому число з моделі")

**Підсумок розділу.** Softmax CLIP ніколи не опускається нижче 1/8 і легко дає 0.5–0.9 порожньому фону — як впевненість це
число непридатне. Сигмоїда SigLIP 2 для фону здебільшого ≈ 0, тож як сигнал вона корисніша, але її абсолютна шкала на аерокропах
зсунута (більшість цілей < 0.1), і «0.5» як поріг не працює й тут. Пороги — лише за виміряною кривою (Демо 3, слайд 49 «Три зони впевненості»).

## 7. Батарея відмов як регресійний тест

П'ять дешевих тестів зі слайдів 37–39 і 46, які варто проганяти після **кожної** зміни моделі, промптів чи ІЧ-конвеєра:
* **заперечення** — топ-20 цілих кадрів за запитом «road without cars»: скільки там справді порожніх доріг;
* **рахунок** — 1, 2, 3, 5, 8 однакових машин на синтетиці і «скільки машин» на реальних кадрах;
* **напис у кадрі** — чи перемагає табличка `AMBULANCE` / `SCHOOL BUS` справжній клас;
* **поворот** — zero-shot macro-recall тих самих кропів, повернутих на 180°, і з TTA (середнє 4 поворотів);
* **полярність ІЧ** — наскільки LWIR і інвертований LWIR далекі від RGB того самого кадру.

In [ ]:
from scipy.stats import spearmanr

frames = P.sample_frames(seqs, per_seq=FRAMES_PER_SEQ)
G = {k: e.encode_images(P.load_images(frames.path)) for k, e in enc.items()}
COUNTS = [1, 2, 3, 5, 8]
WORDS = {1: "one car", 2: "two cars", 3: "three cars", 5: "five cars", 8: "eight cars"}
count_imgs = [P.paste_copies(car, n, seed=n) for n in COUNTS]
BUCKETS = [(0, 0, "no cars"), (1, 3, "one or two cars"), (4, 10, "several cars"), (11, 10**6, "many cars")]
y_b = np.array([next(i for i, (lo, hi, _) in enumerate(BUCKETS) if lo <= c <= hi) for c in frames.cars])
typo_imgs = [P.stamp_text(PLAY["танки, RGB"], t) for t in ("AMBULANCE", "SCHOOL BUS")]
ir_imgs = [PLAY["танки, RGB"], PLAY["танки, LWIR white-hot"], P.transform(PLAY["танки, LWIR white-hot"], invert=True)]
obj_imgs = [crop_imgs[i] for i in np.flatnonzero(~bg)]

battery = {}
for k, e in enc.items():
    top = np.argsort(-(G[k] @ e.text_query("a road without cars")))[:20]
    hits = sum(COUNTS[int(s.argmax())] == n for s, n in zip(e.encode_images(count_imgs) @ np.stack([e.text_query(WORDS[n]) for n in COUNTS]).T, COUNTS))
    pb = (G[k] @ np.stack([e.text_query(t) for *_, t in BUCKETS]).T).argmax(1)
    s_typo = e.encode_images(typo_imgs) @ np.stack([e.text_query(q) for q in ("tanks", "an ambulance", "a school bus")]).T
    T = class_text(e, LABELS, P.AERIAL_TEMPLATES)
    rot = {a: e.encode_images([im.rotate(a, expand=True) for im in obj_imgs]) for a in (0, 90, 180, 270)}
    tta = P.l2n(np.mean(list(rot.values()), 0))
    I_ir = e.encode_images(ir_imgs)
    battery[k] = {
        "заперечення: порожніх у топ-20 «without cars» (у галереї {:.0%})".format((frames.vehicles == 0).mean()): f"{(frames.vehicles.values[top] == 0).sum()}/20",
        "рахунок, синтетика: влучань": f"{hits}/{len(COUNTS)}",
        "рахунок, реальні кадри: macro-recall 4 кошиків (навмання 0.25)": f"{P.macro_recall(y_b, pb):.2f}",
        "Spearman(cos «many cars», к-сть машин)": f"{spearmanr(G[k] @ e.text_query('many cars'), frames.cars)[0]:.2f}",
        "напис перемагає «tanks»": f"{int((s_typo.argmax(1) != 0).sum())}/2",
        "поворот: macro-recall 0° / 180° / TTA": " / ".join(f"{P.macro_recall(y_obj, np.array(LABELS)[(I_ @ T.T).argmax(1)]):.2f}" for I_ in (rot[0], rot[180], tta)),
        "ІЧ: cos(RGB, white-hot) / cos(RGB, black-hot)": f"{I_ir[1] @ I_ir[0]:.2f} / {I_ir[2] @ I_ir[0]:.2f}",
    }
P.show_grid(count_imgs + typo_imgs + ir_imgs, [f"{n} шт" for n in COUNTS] + ["+ AMBULANCE", "+ SCHOOL BUS", "RGB", "LWIR", "LWIR інверт."],
            ncols=10, size=1.6)
display(pd.DataFrame(battery))

**Підсумок розділу.**
* **Заперечення розширює вибірку, а не звужує:** «без машин» шукає машини. Лише позитивні описи; умову «без X» перевіряє детектор.
* **Рахунку немає:** на чистій синтетиці SigLIP 2 вгадує малі числа краще за CLIP, але на реальних кадрах обидві моделі
  ледь кращі за випадковість. Кореляція з кількістю є — це ранжування, а не рахунок. Кількість — з детектора.
* **Напис у кадрі** зсуває відповідь сильніше за будь-яке маскування — для противника це найдешевший обман.
* **Поворот** помітно знижує точність на косих кадрах; TTA повертає частину ціною 4× обчислень (на землі — прийнятно).
* **ІЧ** для вебової моделі — інший домен («чорно-біле фото»): спершу одна полярність і фіксований AGC, потім промпти, потім LoRA.

## 8. Текст чи мітки: zero-shot проти DINOv2

У DINOv2 немає текстової вежі, зате сильні ознаки зображення (Демо 1). Питання: **скільки міток на клас треба DINOv2,
щоб наздогнати zero-shot CLIP без жодної мітки?** Для всіх трьох моделей — лише image tower, ті самі пікселі (кропи 224×224),
kNN і linear probe на 1…20 мітках на клас; train — кропи train, test — val + test-dev (інші відео), 10 наборів міток на бюджет.

In [ ]:
fs_mask = ~bg
FEAT = {k: IMG[k][fs_mask] for k in MODELS}
if USE_DINO:
    dino = P.load_dino(["dinov2"], SIZE, device=DEVICE)
    if dino:
        FEAT["dinov2"] = P.embed_crops(dino["dinov2"], crops[fs_mask], CACHE_DIR / "emb")
yi = np.array([LABELS.index(l) for l in y_all[fs_mask]])
tr = (crops.split[fs_mask] == "train").to_numpy()
te = ~tr
ZS = {k: P.macro_recall(yi[te], (FEAT[k][te] @ class_text(e, LABELS, P.AERIAL_TEMPLATES).T).argmax(1)) for k, e in enc.items()}
FS = P.label_efficiency(FEAT, yi, tr, te, budgets=[1, 2, 5, 10, 20], methods=("per_class",), seeds=range(10))
print(f"train {tr.sum()} кропів · test {te.sum()} кропів (інші відео)")
P.plot_label_efficiency(FS, zero_shot=ZS, xlabel="міток на клас")
T_ = FS.groupby(["clf", "model", "budget"], sort=False).macro_recall.mean().unstack("budget")
T_.insert(0, "zero-shot", [ZS.get(k, np.nan) for _, k in T_.index])
T_.style.format("{:.3f}", na_rep="—").background_gradient(cmap="RdYlGn", vmin=0.3, vmax=0.9)

**Підсумок розділу** (test — лише ~160 кропів з інших відео, тож різниця в кілька пунктів — шум):
* **Нуль міток — територія CLIP / SigLIP.** На цих кропах ні DINOv2, ні власні ознаки CLIP не наздоганяють zero-shot навіть
  на 20 мітках на клас; DINOv2 виходить на рівень zero-shot лише з усіма мітками train (~60 на клас). Для нового типу цілі,
  на який міток ще немає, текст виграє: опис класу словами вартий десятків прикладів.
* **Image tower SigLIP 2 — найсильніший екстрактор ознак тут:** його linear probe наздоганяє власний zero-shot на ~10 мітках
  на клас і обганяє з усіма мітками — «адаптація починається з linear probe, а не з нуля» (слайд 34).
* **CLS-вектор DINOv2 на кропі 224 px не кращий за ознаки CLIP.** Сила DINOv2 — щільні патч-ознаки (Демо 1, розділ 2)
  і ліцензія Apache 2.0, а для класифікації кропів з кількома мітками SigLIP 2 вигідніший.

## Підсумок

| | CLIP ViT-B/16 | SigLIP 2 ViT-B/16 | Що це означає на практиці |
|---|---|---|---|
| Втрата | softmax InfoNCE | сигмоїда на пару | у SigLIP є абсолютна оцінка пари; softmax CLIP для впевненості непридатний |
| Шкала cos | ~0.2–0.35 | ~0.0–0.2 | пороги між моделями не переносяться |
| Розмір ембеддингу | 512 | 768 | індекс і бортовий канал прив'язані до моделі (слайди 50–51) |
| Токенайзер | BPE 49k, 77 токенів | Gemma 256k, 64 токени, канонікалізація | довгі описи обрізаються; важливе — на початку |
| Пастки завантаження | QuickGELU | токенайзер офлайн | модель не падає — просто працює гірше |
| Заперечення, рахунок, написи, ІЧ | ламаються | ламаються | «мішок слів» і вебові дані; метод навчання цього не лікує |

1. Вектори L2-нормовані: скалярний добуток — це сирий cos, і пороги ставимо по ньому, окремо для моделі й промпту.
2. Модель звужує пошук, а не стверджує: 40 годин відео → 200 кадрів для людини (слайди 37, 48).
3. Промпти лише позитивні, з ракурсом, ансамблем; клас «фон» обов'язковий.
4. Порівняння моделей — на своєму домені, з довірчим інтервалом і батареєю відмов як регресійним тестом (слайд 46).

Далі — [Демо 3](lesson05_demo3_video_search.ipynb): пошук по відеоархіву, FAISS, пороги й вузький канал.